In [ ]:
import os

from workshop_helpers import run_environment_check

run_environment_check()

# ROSCon Workshop: Stair-Climbing Policy — Evaluation

This notebook visualizes the generated terrain and compares the same Unitree G1 policy before and
after stair fine-tuning.

Hosted by **AMD and Robotec.ai**, the workshop runs on an **AMD Strix Halo mini-PC**. Evaluation
uses a fixed terrain grid, reset seed, command, and duration so that the checkpoint is the only
variable in the comparison.

## Evaluation goals

* inspect the collision geometry the robot actually encounters;
* quantify where the flat-ground baseline fails;
* connect aggregate results to visible behavior; and
* repeat the identical evaluation with the newly trained checkpoint.

## Setup
The first cell must print `PASS`. This notebook reads the trained policy from the fixed path
`../logs/roscon_stairs/model_stairs.pt`; no timestamp or directory search is required.

In [ ]:
import gslab.tasks  # registers every task  # noqa: F401
from gslab.tasks.registry import load_env_cfg
from workshop_helpers import bootstrap_workshop, print_device_summary, workshop_paths

WORKSHOP_ROOT = bootstrap_workshop()
PATHS = workshop_paths(WORKSHOP_ROOT)
DEVICE = "cuda"  # PyTorch's name for the GPU device; it is the AMD GPU on a ROCm build

TASK = "Unitree-G1-Stairs-Easy"
BASELINE = PATHS.baseline
TRAINED_PATH = PATHS.trained  # written by the training notebook
SUPPLIED_TRAINED_PATH = PATHS.supplied_trained  # a finished 25-minute run, if yours is not there yet
REFERENCE_PATH = PATHS.reference  # 2,048 robots x 200 updates, trained offline

print_device_summary(
    WORKSHOP_ROOT,
    extra={"baseline": BASELINE, "trained": TRAINED_PATH, "reference": REFERENCE_PATH},
)


In [ ]:
from workshop_helpers import EvaluationSession, clear_device_cache, require_checkpoint


## 1. Inspect the generated terrain

The task contains four terrain columns and six difficulty rows. Three columns begin as sampled
heightfields; `real_stairs` uses rigid boxes so Genesis sees actual vertical risers. The images
below show the collision geometry, including a close-up of the hardest stairs.

In [ ]:
from gslab.vis import render_terrain

terrain_cfg = load_env_cfg(TASK, play=True).scene.terrain.terrain_generator
terrain_frames = render_terrain(terrain_cfg, width=860, height=520)
clear_device_cache()


## Build the evaluation scene

- One scene is reused for every measurement. 
- It holds four robots on every terrain cell: six difficulty levels times four terrain types, 96 robots.
- Building the scene also warms up the physics kernels and the the actor and critic policy model runner
- Video playback uses a separate five-robot scene, one robot per difficulty level, recorded by Genesis to MP4.


In [ ]:
evaluation = EvaluationSession(
    task=TASK,
    device=DEVICE,
    workshop_root=WORKSHOP_ROOT,
)
print(evaluation.summary())


## 2. Evaluate the flat-ground baseline

The baseline is given a fixed `0.8 m/s` forward command for 10 seconds, starting from the centre of
its terrain cell, the same spawn the videos use. Ten seconds is enough to cross a whole staircase:
on the box stairs the first riser is 1.25 m out, the top landing spans 3.2–4.2 m and the descent
ends at 5.8 m. Shorter windows only test the first steps and let a slow policy score zero falls.
Falls and forward travel are aggregated by difficulty level across all four terrain types. The
transposed table keeps the levels in columns, making the failure boundary easy to scan. The cell
takes about a minute.


In [ ]:
baseline_rows, baseline_overall = evaluation.evaluate(BASELINE)
evaluation.show_results(baseline_rows, baseline_overall, "before fine-tuning")


### Reference baseline result

| metric | level 0 | level 1 | level 2 | level 3 | level 4 | level 5 | all |
|---|---:|---:|---:|---:|---:|---:|---:|
| falls | 0% | 6% | 12% | 25% | 50% | 50% | **24%** |
| travel | 7.4 m | 7.7 m | 7.5 m | 6.8 m | 5.7 m | 4.9 m | **6.7 m** |

The easiest level is within the flat gait's existing capability. Failures appear from level 1
and reach half of the robots on the two hardest levels, which gives the terrain curriculum a clear
progression to learn. With four robots per cell, one fall is 6% of a level. Exact values vary
slightly between runs because the GPU solver is not bit-for-bit deterministic.


## 3. Watch the baseline fail

The 15-second playback puts one robot on the box staircase of each difficulty level 1 to 5, in a
row, with the same reset seed and the same forward command used for the trained policy. A static
camera behind the row looks down it: the nearest staircase is level 1, the farthest level 5, and
the robots walk away from the camera so the risers stay visible. 
Watch whether the feet lift before the first riser or react only after contact, how that late response
affects torso balance, and at which level the baseline starts to fall.

`watch()` takes two optional arguments:

* `scenario=` — `"upstairs"` (default: box stairs from the run-up, then the landing and the
  descent), `"downstairs"` (the same staircases, but the robots start on the top landing so the
  clip is the descent; the descending risers face the light and are hard to see, follow the
  robots' height instead), or `"terrain"` (the rough-ground column).
* `num_robots=` — with more than six robots, levels repeat.


In [ ]:
evaluation.watch(BASELINE, "flat-ground baseline", duration_s=15.0)
# Other views: scenario="downstairs" or scenario="terrain"


## 4. Evaluate the stair-trained policy

Two checkpoints go through the same measurement as the baseline:

* **Your run** — `logs/roscon_stairs/model_stairs.pt`, written by the training notebook. If it is
  not there yet, the cell falls back to `checkpoints/g1_stairs_easy_trained.pt`, a finished run of
  that notebook with the same 25-minute budget.
* **The reference** — `checkpoints/g1_ref.pt`, trained offline with a larger recipe (2,048 robots,
  200 PPO updates, over an hour on this hardware).

No timestamp or directory search is required.


In [ ]:
TRAINED = TRAINED_PATH if TRAINED_PATH.exists() else SUPPLIED_TRAINED_PATH
TRAINED = require_checkpoint(TRAINED)
REFERENCE = require_checkpoint(REFERENCE_PATH)

print(f"trained  : {TRAINED.relative_to(WORKSHOP_ROOT)}"
      + ("" if TRAINED == TRAINED_PATH else "  (supplied run; your own is not there yet)"))
print(f"reference: {REFERENCE.relative_to(WORKSHOP_ROOT)}")


In [ ]:
trained_rows, trained_overall = evaluation.evaluate(TRAINED)
evaluation.show_results(trained_rows, trained_overall, "after fine-tuning (your run)")


### Evaluate the reference policy

Same scene, same robots, same 10-second rollout: only the checkpoint changes.


In [ ]:
reference_rows, reference_overall = evaluation.evaluate(REFERENCE)
evaluation.show_results(reference_rows, reference_overall, "reference policy")


### Two runs, two recipes

The two policies were trained on the same task, rewards and terrain ladder. They differ in how much
experience PPO saw and how it was collected:

| | Your run (training notebook) | Reference `g1_ref.pt` |
|---|---|---|
| Robots simulated in parallel | 1,024 | 2,048 |
| Control steps per robot per update | 48 | 24 |
| Samples per PPO update | ~49k | ~49k (same) |
| PPO updates | ~115 (25-minute budget) | 200 |
| Total samples | ~5.6 M | ~9.8 M |
| Wall-clock on a Strix Halo | 25 min | over an hour |
| Learning rate, PPO settings | 5e-4, identical | 5e-4, identical |

Per update they are equivalent; the reference simply trained about 1.75× longer. Measured with the
cells above (96 robots, 10 s), against the baseline's 24% falls and 6.5 m:

| metric | your run (typical) | reference |
|---|---:|---:|
| falls, all levels | **~1%** (level 5 only) | **1–3%** (level 5 only) |
| travel in 10 s | 4.3 m (0.45 m/s) | 6.4 m (0.65 m/s) |
| level 5 box stairs, all robots there for 20 s | ~80% fall, at the top of the flight | ~15% fall |

So both move the failure boundary from level 1 to level 5, and the difference is *how* they do it:

* **Speed.** After 25 minutes PPO is in a cautious phase: it has traded speed for not falling and
  advances at about 0.45 m/s on every terrain, flat ground included. Longer training recovers most
  of the commanded speed without giving the robustness back.
* **The hardest staircase.** Your run still stumbles when the level 5 flight ends at the landing;
  the reference climbs it and mostly handles the descent that follows, even though its training
  terrain ended at the landing.

Exact numbers vary between runs: PPO is stochastic and the GPU solver is not bit-for-bit
deterministic. Read **falls and travel together**: falls say whether a policy is robust, travel says
what that robustness costs.


## 5. Watch the improved policy

Only the checkpoint changes. Look for the behavior behind the metrics: earlier foot lift, recovery
from unexpected contact, and a climbing posture that remains controlled across successive steps.
Compare which levels are still standing at the end of the clip against the baseline video, and
watch the top landing and the descent on the far staircases, where the remaining failures happen.


In [ ]:
evaluation.watch(TRAINED, "stair-trained policy", duration_s=15.0)
# Other views: scenario="downstairs" or scenario="terrain"


In [ ]:
evaluation.watch(REFERENCE, "reference policy", duration_s=15.0)
# Other views: scenario="downstairs" or scenario="terrain"


## Key takeaways

* Fixed terrain assignments expose where a policy fails instead of hiding difficulty inside one
  average score.
* Quantitative evaluation and playback answer different questions: how often it fails, and why.
* Fine-tuning should move the failure boundary toward harder terrain without sacrificing the easy
  levels.

> **ROS 2 hint:** If the actor is later wrapped in a controller, reproduce its observation
> ordering, normalization, and 50 Hz control rate exactly.

## Cleanup

Run this cell before opening another GPU-heavy notebook.

In [ ]:
if "evaluation" in globals() and evaluation is not None:
    evaluation.close()
evaluation = None
print("Evaluation scene released.")
